# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [5]:
# imports
import json
import os
import sys

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))


load_dotenv(override=True)

True

In [ ]:
# constants
OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

MODEL_DICT = {
    "gpt": {"gpt-4o-mini", "gpt-5-nano", "gpt-5-mini", "gpt-5.1"},
    "ollama": {"llama3.2", "deepseek-r1:1.5b"},
    "gemini": {"gemini-3-pro-preview", "gemini-flash-latest"},
}


def generic_llm_query(
    system_prompt,
    user_prompt,
    model="gemini-flash-latest",
    stream=False,
    json=False,
    output_response=False,
):
    if model in MODEL_DICT["gpt"]:
        llm = OpenAI()
    elif model in MODEL_DICT["ollama"]:
        llm = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
    elif model in MODEL_DICT["gemini"]:
        llm = OpenAI(
            base_url=GEMINI_BASE_URL,
            api_key=GOOGLE_API_KEY,
        )
    else:
        raise ValueError(f"Model {model} not supported")
    # Create arguments dictionary
    if json:
        system_prompt += " Answer in strict json format."
    kwargs = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }
    # Conditionally add parameters
    if json:
        kwargs["response_format"] = {"type": "json_object"}
    if stream:
        kwargs["stream"] = True
    response = llm.chat.completions.create(**kwargs)

    # display stream as markdown
    if stream:
        output = ""
        if output_response:
            display_handle = display(Markdown(output), display_id=True)
        for chunk in response:
            content = chunk.choices[0].delta.content
            if content:
                output += content
                if output_response:
                    display_handle.update(Markdown(output))
        return output
    # display without stream
    if output_response:
        if json:
            pretty_print_json(response.choices[0].message.content)
        else:
            display(Markdown(response.choices[0].message.content))
    return response.choices[0].message.content


def pretty_print_json(data):
    """
    Prints JSON data in a readable, indented format with syntax highlighting.
    Accepts a dictionary, list, or a JSON string.
    """
    try:
        # If input is a string, try to parse it as JSON first
        if isinstance(data, str):
            data = json.loads(data)

        # Convert back to string with indentation
        pretty_json = json.dumps(data, indent=2, ensure_ascii=False)

        # Display using Markdown for syntax highlighting in the notebook
        display(Markdown(f"```json\n{pretty_json}\n```"))

    except json.JSONDecodeError:
        print("Invalid JSON string provided.")
    except Exception as e:
        print(f"Error prettifying JSON: {e}")

In [ ]:
import os
from typing import Dict, List

from dotenv import load_dotenv

load_dotenv(override=True)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

MODEL_DICT = {
    "gpt": {"gpt-4o-mini", "gpt-5-nano", "gpt-5-mini", "gpt-5.1", "gpt-5.2"},
    "ollama": {"llama3.2", "deepseek-r1:1.5b"},
    "gemini": {"gemini-3-pro-preview", "gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-flash-latest","gemini-flash-lite-latest"},
}


def pretty_print_json(data):
    """
    Prints JSON data in a readable, indented format with syntax highlighting.
    Accepts a dictionary, list, or a JSON string.
    """
    try:
        # If input is a string, try to parse it as JSON first
        if isinstance(data, str):
            data = json.loads(data)

        # Convert back to string with indentation
        pretty_json = json.dumps(data, indent=2, ensure_ascii=False)

        # Display using Markdown for syntax highlighting in the notebook
        display(Markdown(f"```json\n{pretty_json}\n```"))

    except json.JSONDecodeError:
        print("Invalid JSON string provided.")
    except Exception as e:
        print(f"Error prettifying JSON: {e}")


class LLMQuery:
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

    def __init__(
        self,
        system_prompt: str,
        model="gemini-flash-lite-latest",
        stream=False,
        json_format=False,
    ):
        self.model = model
        self.stream = stream
        self.json_format = json_format
        self.client = self.get_client()
        self.system_prompt = system_prompt
        self.chat_history: List[Dict[str, str]] = []
        self.response = ""

    def get_client(self):
        if self.model in MODEL_DICT["gpt"]:
            client = OpenAI()
        elif self.model in MODEL_DICT["ollama"]:
            client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
        elif self.model in MODEL_DICT["gemini"]:
            client = OpenAI(
                base_url=GEMINI_BASE_URL,
                api_key=GOOGLE_API_KEY,
            )
        else:
            raise ValueError(f"Model {self.model} not supported")
        return client

    def query(
        self,
        user_prompt,
        use_history=False,
        display_output=False,
        stream=None,
        json_format=None,
    ):
        if stream is None:
            stream = self.stream
        if json_format is None:
            json_format = self.json_format

        messages = [{"role": "system", "content": self.system_prompt}]
        if use_history:
            messages.extend(self.chat_history)
        messages.append({"role": "user", "content": user_prompt})
        kwargs = {"model": self.model, "messages": messages}
        if json_format:
            kwargs["response_format"] = {"type": "json_object"}
        if stream:
            kwargs["stream"] = True
        response = self.client.chat.completions.create(**kwargs)

        # stream output
        if stream:
            output = ""
            if display_output:
                display_handle = display(Markdown(output), display_id=True)
            for chunk in response:
                content = chunk.choices[0].delta.content
                if content:
                    output += content
                    if display_output:
                        display_handle.update(Markdown(output))
            self.response = output
        else:
            # without stream
            self.response = response.choices[0].message.content
            if display_output:
                self.display_response()

        self.chat_history.append({"role": "user", "content": user_prompt})
        self.chat_history.append({"role": "assistant", "content": self.response})

    def display_response(self):
        if self.json_format:
            pretty_print_json(self.response)
        else:
            display(Markdown(self.response))

    def get_chat_history_as_string(self):
        history: List[str] = []
        for msg in self.chat_history:
            role = msg["role"].capitalize()
            content = msg["content"]
            if role == "User":
                history.append(f"**User**: {content}")
            elif role == "Assistant":
                history.append(f"**Assistant**: {content}")
        return "\n\n".join(history)

    def display_chat_history(self):
        display(Markdown(self.get_chat_history_as_string()))

In [54]:
c = LLMQuery(system_prompt="You are a helpful assistant", stream=True)
c.query("Hello, my name is Martin", display_output=True)

Hello Martin, how can I help you today?

In [55]:
c.chat_history

[{'role': 'user', 'content': 'Hello, my name is Martin'},
 {'role': 'assistant', 'content': 'Hello Martin, how can I help you today?'}]

In [58]:
c.query("Whats my name?", use_history=True)
Markdown(c.get_chat_history_as_string())

**User**: Hello, my name is Martin

**Assistant**: Hello Martin, how can I help you today?

**User**: Whats my name?

**Assistant**: Your name is Martin.

**User**: Whats my name?

**Assistant**: As an AI, I don't have access to your personal information, so I don't know your name.

If you tell me your name, I'll be happy to remember it!

**User**: Whats my name?

**Assistant**: Your name is Martin. I remember you told me earlier!

In [16]:
display(Markdown(c.get_chat_history_as_string()))

User: Hello, my name is Martin


Hello Martin! It's a pleasure to meet you. How may I assist you today?

User: Whats my name?


Assistant: I don't have access to your personal information, including your name. Is there anything else I can help you with?

User: Hello, my name is Martin


Hello Martin! It's a pleasure to meet you. How may I assist you today?

User: Whats my name?


Assistant: I don't have access to your personal information, including your name. Is there anything else I can help you with?

User: Whats my name?


Your name is Martin, as you told me earlier.

In [6]:
# Simple Test
display(Markdown(generic_llm_query("You are a helpful assistant.", "Who are you?")))

I am a large language model, trained by Google.

In [7]:
# set up environment

SYSTEM_PROMPT_TECHNICAL_EXPERT = """
You are a technical expert with deep knowledge in software development and programming languages. You are able to answer questions about programming languages, frameworks, and tools. You are also able to answer questions about the latest trends in software development and programming languages.
"""

In [8]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [14]:
expert = LLMQuery(system_prompt=SYSTEM_PROMPT_TECHNICAL_EXPERT, stream=True, model = "gemini-flash-latest")
expert.query(question, display_output=True, stream=True)

This concise line of Python code leverages several advanced language features—specifically **set comprehensions** and the **`yield from`** expression—to efficiently extract and emit unique author names from a collection of book data.

Here is a detailed explanation of what the code does and why it is written this way.

---

## 1. Context: Where This Code Must Reside

It is crucial to understand that the `yield from` expression can **only** be used inside a **generator function** (a function that contains `yield` statements, making it an iterator).

The overall goal of the function containing this line is to act as a **generator**, producing a stream of authors, one at a time.

## 2. What the Code Does (Step-by-Step Breakdown)

The code executes in two main phases:

### Phase 1: The Set Comprehension (Processing `books`)

The internal part, `{book.get("author") for book in books if book.get("author")}`, is a **Set Comprehension**. Assuming `books` is an iterable of dictionaries (e.g., `list[dict]`), this comprehension performs the following:

1.  **Iteration:** It iterates over every `book` in the `books` iterable.
2.  **Robust Key Access:** It uses `book.get("author")`. This is robust because if a dictionary is missing the `"author"` key, `.get()` returns `None` instead of raising a `KeyError`.
3.  **Filtering:** The `if book.get("author")` clause filters the results. Since `None` is considered falsy, this ensures that only books where the `"author"` key exists and has a "truthy" value (i.e., not `None`, not an empty string `""`) are included.
4.  **Uniqueness:** Because the result is a **set** (indicated by the curly braces `{}`), all duplicate author names are automatically eliminated.

**Result of Phase 1:** A complete, in-memory Python `set` containing all unique, non-empty author names found in the input list `books`.

### Phase 2: The `yield from` Expression (Streaming the Authors)

The `yield from` expression delegates the iteration process to the iterable object that follows it (in this case, the newly created set of unique authors).

1.  **Delegation:** `yield from` takes the authors set and begins iterating over it.
2.  **Streaming:** It effectively acts as a loop: for every author in the set, it `yields` that author back to the caller of the generator function.

**Overall Function:** The code efficiently extracts a unique collection of authors and then streams them one by one.

## 3. Why This Code is Written This Way (Design Rationale)

This construction is highly idiomatic Python, primarily favored for **conciseness, efficiency, and clarity** in generator functions.

### A. Guaranteeing Uniqueness Efficiently

The primary reason for using a **set comprehension** (vs. a list comprehension or generator expression) is to ensure that the output stream contains only unique authors.

*   If the developer had used a list comprehension, they would need a secondary step (like converting the list to a set) to deduplicate the results, adding complexity.
*   By using the set comprehension, the deduplication is handled automatically and very efficiently by the internal hashing mechanics of the Python `set` structure.

### B. Efficiency and Clarity with `yield from` (Python 3.3+)

The `yield from` expression replaces a more verbose pattern:

```python
def get_unique_authors(books):
    # 1. Compute the set
    unique_authors = {book.get("author") for book in books if book.get("author")}

    # 2. Iterate and yield explicitly
    for author in unique_authors:
        yield author
```

The single line using `yield from` achieves the exact same result while being more compact and often slightly faster because `yield from` handles the iteration delegation directly within the interpreter.

### C. Robust Data Handling

Using the dictionary method `.get("author")` prevents program crashes (`KeyError`) if the input data (`books`) is inconsistent (i.e., some dictionaries are missing the expected "author" key). This makes the code highly robust against messy data sources.

In [13]:
# Get gpt-4o-mini to answer, with streaming

expert = LLMQuery(system_prompt=SYSTEM_PROMPT_TECHNICAL_EXPERT, stream=True, model = "gemini-flash-lite-latest")
expert.query(question, display_output=True, stream=True)
# display in markdown

This code snippet is a concise and powerful way to generate a sequence of author names from a list of book dictionaries, utilizing Python's **generator expressions** and the `yield from` statement.

Here is a detailed breakdown of what it does, why it's structured this way, and the concepts involved.

---

## 1. Breakdown of the Code

The entire line is a **generator function** definition (or more accurately, the body of a generator function, as it uses `yield from`).

Let's break it down from the inside out:

### A. The Inner Part: Generator Expression

```python
{book.get("author") for book in books if book.get("author")}
```

This is a **set comprehension** (because of the curly braces `{}`) used as a generator expression (though technically, comprehensions create a collection, in this context, it's often used to generate the sequence to be consumed by `yield from`).

1.  **`for book in books`**: It iterates over a collection named `books`. We assume `books` is an iterable (like a list) where each element (`book`) is a dictionary.
2.  **`if book.get("author")`**: This is a filter. It checks if the key `"author"` exists *and* if the value associated with that key is truthy (not `None`, not an empty string, etc.).
3.  **`book.get("author")`**: If the condition passes, it retrieves the value associated with the `"author"` key.
4.  **Result**: This part produces a set containing all the unique, non-empty author names found in the `books` list.

**Crucially, in this specific context, using a set comprehension `{...}` here is slightly unusual if the goal is just sequential yielding, but it works.** The set comprehension first builds the entire set of unique authors in memory, and *then* the `yield from` iterates over that completed set.

### B. The Outer Part: `yield from`

```python
yield from ...
```

The `yield from` expression is used in a generator function to delegate part of the execution to another generator or iterable.

1.  **Delegation**: It effectively says, "I want to yield every item that the iterable on the right-hand side yields, one by one, as if I were yielding them directly."
2.  **Efficiency (when applied to a true generator)**: When the right-hand side is another generator, `yield from` passes values directly from the sub-generator to the caller of the main generator, avoiding intermediate storage and often making the delegation more efficient than a manual loop with `yield`.

### C. Putting It Together (What the Full Line Does)

Assuming this line is inside a function defined with `def some_generator():`, the entire function does the following:

1.  It first processes the `books` list to find all the authors.
2.  It collects these authors into a **set** (ensuring uniqueness).
3.  It then iterates over this resulting set, **yielding each unique author name** individually to the caller of the generator function.

---

## 2. Why Use This Construction? (The Rationale)

This construction is primarily used to achieve **concise, unique iteration.**

### A. Conciseness (vs. Standard Loop)

The equivalent standard loop structure would look like this:

```python
def get_unique_authors(books):
    authors_set = set()
    for book in books:
        author = book.get("author")
        if author:
            authors_set.add(author)

    for author in authors_set:
        yield author
```

The original single line replaces this multi-step process:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

### B. Ensuring Uniqueness

By using the **set comprehension** (`{...}`), the code automatically handles duplicate authors. If the book list contains ten books by "Jane Doe," the resulting set will only contain "Jane Doe" once.

### C. Why `yield from` instead of `yield`?

If the code were:

```python
yield {book.get("author") for book in books if book.get("author")}
```

This would **yield only one item**: the entire set object itself.

Since the goal is to yield **each author name individually**, `yield from` is necessary to iterate over the contents of the set and yield each element separately.

---

## Summary

The code snippet is a compact Pythonic way to create a **generator that yields every unique, non-empty author name** found within a list of book dictionaries. It leverages the power of set comprehensions for aggregation and filtering, combined with `yield from` for efficient sequential output.

In [40]:
expert.display_response()

**Assistant**: This single line of Python code is extremely dense, utilizing a **Set Comprehension** and the **`yield from`** keyword, indicating it is running inside a **Generator Function**.

Here is a detailed explanation of what the code does and why it is structured this way.

---

## 1. What the Code Does (The Process)

This code performs three distinct operations: filtering, de-duplication, and delegated yielding.

### Step 1: Set Comprehension (Filtering and De-duplication)

The core structure is a Python **Set Comprehension**:

```python
{book.get("author") for book in books if book.get("author")}
```

1.  **Iteration (`for book in books`):** It assumes `books` is an iterable (like a list) containing dictionaries, where each dictionary represents a book.
2.  **Filtering (`if book.get("author")`):** It checks the retrieved author value.
    *   `book.get("author")` safely tries to retrieve the value for the `"author"` key (returning `None` if the key is missing).
    *   In a boolean context, `None` or an empty string (`""`) evaluates to `False`. This ensures that only books that have a defined, non-empty author value are processed.
3.  **Collection (Set Creation):** The resulting values (the author names) are collected into a **Set** (indicated by the curly braces `{}`).

**Result of Step 1:** An entire Python `set` object is created in memory, containing every unique author name from the input `books` list that passed the filter.

### Step 2: Yield Delegation (`yield from`)

The `yield from` statement takes the set created in Step 1 and delegates iteration to it.

```python
yield from [the resulting set]
```

Instead of the function writing a loop like:

```python
for item in the_set:
    yield item
```

`yield from` handles this iteration internally and efficiently. It yields each unique author name, one by one, to the caller of the generator function.

## 2. Why it Works This Way (Purpose and Benefits)

This structure is used to achieve two primary goals concisely: **guaranteed uniqueness** and **generator delegation**.

### A. The Primary Goal: Guaranteed Uniqueness

The core reason for using the **Set Comprehension** (curly braces `{}`) is to efficiently guarantee that the output stream contains only unique author names.

*   If the input list `books` had 10 entries for "Jane Austen," the resulting set would only contain "Jane Austen" once.
*   Using a set comprehension is generally faster and more Pythonic for de-duplication than sorting a list and manually checking for duplicates.

### B. Efficiency and Clarity with `yield from`

`yield from` simplifies the code structure. It signifies that the generator function is delegating its responsibilities to iterate over another iterable (the set of authors). This provides a cleaner syntax compared to writing an explicit internal loop.

### C. Filtering and Safety

The use of `.get("author")` provides a safe way to access dictionary keys without raising a `KeyError` if the key is missing. The subsequent check ensures that the yielded results are meaningful (not `None` or an empty string).

## 3. Important Context: Materialization vs. Laziness

It is critical to note the difference between a Set Comprehension and a Generator Expression:

| Feature | Set Comprehension (`{...}`) | Generator Expression (`(...)`) |
| :--- | :--- | :--- |
| **Materialization** | **Eager.** The entire set is built in memory *before* yielding starts. | **Lazy.** Items are processed and yielded one at a time. |
| **Goal** | Efficient de-duplication (Uniqueness). | Memory efficiency and immediate start of processing. |

Because the code uses curly braces (`{...}`), the *entire* list of unique authors must be calculated and stored in memory as a set before the first result can be `yield`ed. This structure prioritizes **uniqueness** over strict **laziness** for this specific step. If uniqueness were not required, the efficient, lazy alternative would be a Generator Expression:

```python
# A lazy generator expression (no uniqueness guarantee)
yield from (book.get("author") for book in books if book.get("author")) 
```

In [ ]:
# Get Llama 3.2 to answer
stream_response = generic_llm_query(
    SYSTEM_PROMPT_TECHNICAL_EXPERT,
    question,
    model="llama3.2",
    stream=False,
    output_reponse=True,
    json=True,
)

{
  "type": "Code Explanation",
  "description": "Declarative Code with Yield From",
  "code": "[{\"line\":1,\"code\":\"yield from {book.get(\\\"author\\\") for book in books if book.get(\\\"author\")}\\\\n\\",", ":", "  }